In [ ]:
import sys
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    make_scorer,
    f1_score,
)
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    LabelEncoder,
    PolynomialFeatures,
    StandardScaler,
)


In [ ]:
REPO_PATH = Path('/content/course_paper')

!git clone --depth 1 --filter=blob:none --sparse https://github.com/incRED1bl/course_paper.git {REPO_PATH}
!git -C {REPO_PATH} sparse-checkout set colab/data/respiratory_features.csv

sys.path.insert(0, str(REPO_PATH))

!cp {REPO_PATH}/colab/data/respiratory_features.csv ../../data/respiratory_features.csv

In [ ]:
df = pd.read_csv('/content/course_paper/colab/data/respiratory_features.csv')
df.head()

In [ ]:
df = df.drop(columns=['filename', 'patient_id'])
df = df[df['diagnosis'] != 'Unknown']
df.head()

In [ ]:
def map_diagnosis(x):
    if x == 'Healthy':
        return 'Healthy'
    elif x == 'COPD':
        return 'COPD'
    else:
        return 'Disease'

df['diagnosis'] = df['diagnosis'].apply(map_diagnosis)
df.head()

In [ ]:
X = df.drop(columns=['diagnosis'])
y_raw = df['diagnosis']

In [ ]:
le = LabelEncoder()
y = le.fit_transform(y_raw)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
X_train = X_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)
y_train = np.asarray(y_train)
y_test  = np.asarray(y_test)

In [ ]:
!pip install optuna
# !rm respiratory_optuna.db
scorer = make_scorer(f1_score, average='macro')

def objective(trial):
    C = trial.suggest_float("C", 1e-3, 100, log=True)
    tol = trial.suggest_float("tol", 1e-6, 1e-2, log=True)
    max_iter = trial.suggest_int("max_iter", 200, 2000, step=100)

    solver = trial.suggest_categorical("solver", ["lbfgs", "liblinear", "saga"])

    penalty = "l2"
    if solver in ["liblinear", "saga"]:
        penalty = trial.suggest_categorical("penalty", ["l1", "l2", "elasticnet"])

    l1_ratio = None
    if penalty == "elasticnet":
        l1_ratio = trial.suggest_float("l1_ratio", 0.0, 1.0)

    class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])

    poly_degree = trial.suggest_int("poly_degree", 1, 2)

    prep = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("poly", PolynomialFeatures(degree=poly_degree, include_bias=False)),
        ("scaler", StandardScaler()),
    ])

    clf = LogisticRegression(
        C=C,
        tol=tol,
        max_iter=max_iter,
        solver=solver,
        penalty=penalty,
        l1_ratio=l1_ratio,
        class_weight=class_weight,
        random_state=42,
        n_jobs=-1
    )

    model = Pipeline([("prep", prep), ("clf", clf)])

    try:
        scores = cross_val_score(
            model,
            X_train,
            y_train,
            cv=StratifiedKFold(n_splits=4, shuffle=True, random_state=42),
            scoring=scorer,
            n_jobs=-1
        )
        return scores.mean()
    except Exception as e:
        print(f"Trial failed: {e}")
        return 0.0


study = optuna.create_study(
    direction="maximize",
    study_name="respiratory_logreg_macrof1",
    storage="sqlite:///respiratory_optuna.db",
    load_if_exists=True
)

study.optimize(
    objective,
    n_trials=60,
    timeout=5400,
    show_progress_bar=True
)

best_result = {
    "best_macro_f1": round(study.best_value, 5),
    **study.best_params
}

print("\nBest macro-F1:", best_result["best_macro_f1"])
print("best_params = {")
for k, v in best_result.items():
    if k == "best_macro_f1":
        continue
    else:
        if isinstance(v, str):
            print(f"    '{k}': '{v}',")
        elif isinstance(v, (int, float)):
            print(f"    '{k}': {v},")
        else:
            print(f"    '{k}': {v!r},")
print("}")


In [ ]:
best_params = {
    'C': 3.3024645457052553,
    'tol': 1.3406593880770292e-06,
    'max_iter': 400,
    'solver': 'lbfgs',
    'class_weight': None,
    'poly_degree': 2,
}

In [ ]:
best_poly_degree = best_result['poly_degree']

prep = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("poly", PolynomialFeatures(degree=best_poly_degree, include_bias=False)),
    ("scaler", StandardScaler()),
])

In [ ]:
best_clf = LogisticRegression(
    C=best_result['C'],
    tol=best_result['tol'],
    max_iter=best_result['max_iter'],
    solver=best_result['solver'],
    penalty=best_result.get('penalty', 'l2'),
    l1_ratio=best_result.get('l1_ratio', None),
    class_weight=best_result.get('class_weight', None),
    random_state=42,
    n_jobs = 1 if best_result['solver'] == 'liblinear' else -1
)

In [ ]:
model = Pipeline([
    ("prep", prep),
    ("clf", best_clf)
])

In [ ]:
model["prep"].fit_transform(X_train).shape

In [ ]:
model

In [ ]:
oof_pred = np.zeros(len(y_train), dtype=np.int64)
fold_acc = []
fold_auc_macro = []
fold_auc_weighted = []

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (tr_idx, val_idx) in enumerate(cv.split(X_train, y_train), 1):
    print(f"Fold {fold}/5 ... ", end="")

    m = clone(model)
    m.fit(X_train.iloc[tr_idx], y_train[tr_idx])

    proba = m.predict_proba(X_train.iloc[val_idx])
    pred = m.classes_[np.argmax(proba, axis=1)]

    oof_pred[val_idx] = pred

    acc = accuracy_score(y_train[val_idx], pred)
    auc_macro   = roc_auc_score(y_train[val_idx], proba, multi_class="ovr", average="macro")
    auc_weighted = roc_auc_score(y_train[val_idx], proba, multi_class="ovr", average="weighted")

    fold_acc.append(acc)
    fold_auc_macro.append(auc_macro)
    fold_auc_weighted.append(auc_weighted)

    print(f"ACC={acc:.4f} | AUC(macro)={auc_macro:.4f} | AUC(w)={auc_weighted:.4f}")

In [ ]:
print("\n" + "─"*70)
print("Cross-validation results (5 folds)")
print("─"*70)
print(f"Accuracy mean     : {np.mean(fold_acc):.4f} ± {np.std(fold_acc):.4f}")
print(f"AUC macro mean    : {np.mean(fold_auc_macro):.4f} ± {np.std(fold_auc_macro):.4f}")
print(f"AUC weighted mean : {np.mean(fold_auc_weighted):.4f} ± {np.std(fold_auc_weighted):.4f}")
print("─"*70)

print("\nOOF Classification Report (full training set):")
print(classification_report(
    y_train,
    oof_pred,
    target_names=le.classes_,
    digits=4
))

In [ ]:
model.fit(X_train, y_train)
test_pred = model.predict(X_test)
test_acc = accuracy_score(y_test, test_pred)
print("Test ACC:", test_acc)